# LineFormer probe — batch chart→data on Colab GPU (v2)

**Goal:** judge published-LineFormer quality on OUR catalysis figures before
investing in a standalone rewrite. Free T4 GPU; ~20 min end to end.

Works in BOTH the Colab web UI and the **VS Code Colab-kernel extension**
(file transfer uses an ipywidgets uploader, not the Colab web frontend).
When creating a VS Code Colab server, pick a **T4 GPU** shape (default is CPU).

Run top to bottom. The only interactive step is cell 6 (select the 30 probe
PNGs from `figures/lineformer_probe/` on your machine).

In [ ]:
# 1. GPU check
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — pick a T4 runtime/server shape'

In [ ]:
# 2. Clone the pipeline repo (LineFormer + ChartDete wrapper)
%cd /content
!rm -rf extract-line-chart-data
!git clone -q https://github.com/tdsone/extract-line-chart-data.git
%cd extract-line-chart-data
!ls

In [ ]:
# 3. Install — mmcv/mmdet pinned to THIS Colab's torch up front (~5-10 min).
import torch
cu = 'cu' + torch.version.cuda.replace('.', '')
tv = torch.__version__.split('+')[0]
print(f'pinning mmcv for torch{tv}/{cu}')
!pip install -q openmim mmengine ipywidgets
!mim install -q "mmcv>=2.0.0,<2.2.0"
!pip install -q -e ".[local]"
!bash setup_local_env.sh || echo '--- setup_local_env.sh reported errors: read them before continuing ---'

### 4. Only if the install fights the current torch
```python
!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
```
then **restart the session** and re-run from cell 2 (skip the pin in cell 3).

In [ ]:
# 5. Pre-download the model weights (the wrapper lazy-downloads them from
#    HuggingFace into ~/.cache/plextract/ on first inference — fetch them now
#    so progress is visible and network failures surface early)
from huggingface_hub import snapshot_download
from pathlib import Path
for repo, sub in [("tdsone/lineformer", "lineformer"),
                  ("tdsone/chartdete", "chartdete")]:
    d = Path.home() / ".cache" / "plextract" / sub
    if not d.exists():
        print(f"downloading {repo} ...")
        snapshot_download(repo, local_dir=str(d))
    print(sub, "->", sorted(p.name for p in d.iterdir()))
import glob, os
assert glob.glob(os.path.expanduser("~/.cache/plextract/*/*.pth")), "weights missing"
print("checkpoints ready")

In [ ]:
# 6. Upload the probe images — ipywidgets uploader (works in VS Code too).
#    Click the button and MULTI-SELECT all PNGs from figures/lineformer_probe/
#    on your machine (raw PNGs or the zip — both accepted). Then run cell 7.
import ipywidgets as widgets
from IPython.display import display
uploader = widgets.FileUpload(accept='.png,.zip', multiple=True,
                              description='Select files')
display(uploader)
print('select the probe files above, THEN run the next cell')

In [ ]:
# 7. Collect the input set (priority: uploader > leftovers on the VM >
#    Drive-link fallback). Idempotent — safe to re-run.
import os, glob, shutil, zipfile, io
os.makedirs('input', exist_ok=True)

def _ingest_zip_bytes(b):
    with zipfile.ZipFile(io.BytesIO(b)) as z:
        z.extractall('input')

n0 = len(glob.glob('input/*.png'))
# a) the uploader widget
try:
    items = uploader.value
    pairs = ([(v['name'], bytes(v['content'])) for v in items]
             if isinstance(items, (list, tuple))
             else [(k, bytes(v['content'])) for k, v in items.items()])
    for name, content in pairs:
        if name.endswith('.zip'):
            _ingest_zip_bytes(content)
        elif name.endswith('.png'):
            open(os.path.join('input', os.path.basename(name)), 'wb').write(content)
except NameError:
    pass
# b) leftovers from an earlier gdown folder download on this VM
for p in glob.glob('drive_in/**/*.png', recursive=True):
    dst = os.path.join('input', os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)
# c) LAST resort: anonymous Drive fetch (public-link quota applies!)
if not glob.glob('input/*.png'):
    DRIVE_LINK = 'https://drive.google.com/drive/folders/1YwfgY43hBMLESwZZrYpKiiKFXYPC1CCL?usp=sharing'
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download_folder(url=DRIVE_LINK, output='drive_in', quiet=False)
    for p in glob.glob('drive_in/**/*.png', recursive=True):
        shutil.copy(p, os.path.join('input', os.path.basename(p)))

pngs = glob.glob('input/*.png')
print(f'input/: {len(pngs)} images  (+{len(pngs) - n0} new this run)')
assert pngs, 'no input images — use the uploader in cell 6'

In [ ]:
# 8. Batch extract (single-panel *__p?.png crops are the intended input;
#    the *__full.png images show how it copes with composites)
from plextract import extract
extract(input_dir='input', output_dir='output', backend='local')
!find output -name '*.json' | head -20

In [ ]:
# 9. Package the results and get them back to your machine.
import os, shutil, base64
shutil.make_archive('lineformer_probe_results', 'zip', 'output')
size = os.path.getsize('lineformer_probe_results.zip')
print(f'results zip: {size/1e6:.1f} MB')
done = False
try:                                   # Colab web frontend
    from google.colab import files
    files.download('lineformer_probe_results.zip')
    done = True
except Exception as e:
    print('files.download unavailable:', e)
if not done:
    try:                               # your own Drive (no public-link quota)
        from google.colab import drive
        drive.mount('/content/drive')
        shutil.copy('lineformer_probe_results.zip',
                    '/content/drive/MyDrive/lineformer_probe_results.zip')
        print('-> saved to Drive: MyDrive/lineformer_probe_results.zip')
        done = True
    except Exception as e:
        print('Drive mount unavailable:', e)
if not done and size < 20e6:           # inline download link (VS Code renderer)
    from IPython.display import HTML, display
    b64 = base64.b64encode(open('lineformer_probe_results.zip', 'rb').read()).decode()
    display(HTML(f'<a download="lineformer_probe_results.zip" '
                 f'href="data:application/zip;base64,{b64}">'
                 f'click to download lineformer_probe_results.zip</a>'))
print('drop the zip on the repo machine as figures/lineformer_probe_results.zip')

## What happens next (on the repo machine)
Claude fuses the LineFormer pixel traces with the existing axis-calibration
layer and rebuilds the verification HTML — original | LineFormer | CV reader,
judged by eye.

**Judging criteria:** (1) right number of curves per panel — especially the
GRAYSCALE marker figures (catcom_2009, jcat_2015.01) where colour-based CV is
blind; (2) traces follow the printed curves through crossings; (3) composite
(*__full*) behaviour vs pre-split panels.